# Projeto 9: Regressão carros usados - validação cruzada

## Etapa 1: Importação das bibliotecas



In [1]:
#!pip install skorch

In [2]:
import pandas as pd
import torch.nn as nn
from skorch import NeuralNetRegressor
import torch
from sklearn.model_selection import cross_val_score
import time
torch.__version__

'2.8.0+cu128'

## Etapa 2: Base de dados

In [5]:
# semente inicial
torch.manual_seed(123)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [6]:
base = pd.read_csv('autos.csv', encoding = 'ISO-8859-1')
display(base.tail())

,dateCrawled,name,seller,offerType,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,kilometer,monthOfRegistration,fuelType,brand,notRepairedDamage,dateCreated,nrOfPictures,postalCode,lastSeen
371523,2016-03-14 17:48:27,Suche_t4___vito_ab_6_sitze,privat,Angebot,2200,test,NaN,2005,NaN,0,NaN,20000,1,NaN,sonstige_autos,NaN,2016-03-14 00:00:00,0,39576,2016-04-06 00:46:52
371524,2016-03-05 19:56:21,Smart_smart_leistungssteigerung_100ps,privat,Angebot,1199,test,cabrio,2000,automatik,101,fortwo,125000,3,benzin,smart,nein,2016-03-05 00:00:00,0,26135,2016-03-11 18:17:12
371525,2016-03-19 18:57:12,Volkswagen_Multivan_T4_TDI_7DC_UY2,privat,Angebot,9200,test,bus,1996,manuell,102,transporter,150000,3,diesel,volkswagen,nein,2016-03-19 00:00:00,0,87439,2016-04-07 07:15:26
371526,2016-03-20 19:41:08,VW_Golf_Kombi_1_9l_TDI,privat,Angebot,3400,test,kombi,2002,manuell,100,golf,150000,6,diesel,volkswagen,NaN,2016-03-20 00:00:00,0,40764,2016-03-24 12:45:21
371527,2016-03-07 19:39:19,BMW_M135i_vollausgestattet_NP_52.720____Euro,privat,Angebot,28990,control,limousine,2013,manuell,320,m_reihe,50000,8,benzin,bmw,nein,2016-03-07 00:00:00,0,73326,2016-03-22 03:17:10


In [7]:
# apagar alguns atributos que não são muito relevantes.


# datas no momento não nos interessa
#'dateCrawled',
#'dateCreated',
#'nrOfPictures', número de fotos não interfere
#'postalCode' no momento o cep não importa
#  'lastSeen' data da última visualização no site.

In [8]:
base = base.drop('dateCrawled', axis = 1)
base = base.drop('dateCreated', axis = 1)
base = base.drop('nrOfPictures', axis = 1)
base = base.drop('postalCode', axis = 1)
base = base.drop('lastSeen', axis = 1)
base = base.drop('name', axis = 1)
base = base.drop('seller', axis = 1)
base = base.drop('offerType', axis = 1)

In [9]:
# retirar os outliers
base = base[base.price > 10]
base = base.loc[base.price < 350000]

In [10]:
display(base.tail())

,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,kilometer,monthOfRegistration,fuelType,brand,notRepairedDamage
371523,2200,test,NaN,2005,NaN,0,NaN,20000,1,NaN,sonstige_autos,NaN
371524,1199,test,cabrio,2000,automatik,101,fortwo,125000,3,benzin,smart,nein
371525,9200,test,bus,1996,manuell,102,transporter,150000,3,diesel,volkswagen,nein
371526,3400,test,kombi,2002,manuell,100,golf,150000,6,diesel,volkswagen,NaN
371527,28990,control,limousine,2013,manuell,320,m_reihe,50000,8,benzin,bmw,nein


In [11]:
# preencher os dados nulos

valores = {'vehicleType': 'limousine', 'gearbox': 'manuell',
           'model': 'golf', 'fuelType': 'benzin',
           'notRepairedDamage': 'nein'}
base = base.fillna(value = valores)

In [12]:
previsores = base.iloc[:, 1:13].values
preco_real = base.iloc[:, 0].values.reshape(-1, 1)

In [14]:
list(base.columns[1:13]) # previsores

['abtest',
 'vehicleType',
 'yearOfRegistration',
 'gearbox',
 'powerPS',
 'model',
 'kilometer',
 'monthOfRegistration',
 'fuelType',
 'brand',
 'notRepairedDamage']

In [17]:
list(base.columns[:1]) # preço real

['price']

In [18]:
previsores

array([['test', 'limousine', 1993, ..., 'benzin', 'volkswagen', 'nein'],
       ['test', 'coupe', 2011, ..., 'diesel', 'audi', 'ja'],
       ['test', 'suv', 2004, ..., 'diesel', 'jeep', 'nein'],
       ...,
       ['test', 'bus', 1996, ..., 'diesel', 'volkswagen', 'nein'],
       ['test', 'kombi', 2002, ..., 'diesel', 'volkswagen', 'nein'],
       ['control', 'limousine', 2013, ..., 'benzin', 'bmw', 'nein']],
      dtype=object)

In [19]:
preco_real

array([[  480],
       [18300],
       [ 9800],
       ...,
       [ 9200],
       [ 3400],
       [28990]])

In [20]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
onehotencoder = ColumnTransformer(transformers = [("OneHot", OneHotEncoder(), [0,1,3,5,8,9,10])], remainder = 'passthrough')
previsores = onehotencoder.fit_transform(previsores).toarray()

In [21]:
previsores

array([[0.00e+00, 1.00e+00, 0.00e+00, ..., 0.00e+00, 1.50e+05, 0.00e+00],
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 1.90e+02, 1.25e+05, 5.00e+00],
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 1.63e+02, 1.25e+05, 8.00e+00],
       ...,
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 1.02e+02, 1.50e+05, 3.00e+00],
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 1.00e+02, 1.50e+05, 6.00e+00],
       [1.00e+00, 0.00e+00, 0.00e+00, ..., 3.20e+02, 5.00e+04, 8.00e+00]])

In [22]:
previsores = previsores.astype('float32')
preco_real = preco_real.astype('float32')

In [23]:
previsores

array([[0.00e+00, 1.00e+00, 0.00e+00, ..., 0.00e+00, 1.50e+05, 0.00e+00],
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 1.90e+02, 1.25e+05, 5.00e+00],
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 1.63e+02, 1.25e+05, 8.00e+00],
       ...,
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 1.02e+02, 1.50e+05, 3.00e+00],
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 1.00e+02, 1.50e+05, 6.00e+00],
       [1.00e+00, 0.00e+00, 0.00e+00, ..., 3.20e+02, 5.00e+04, 8.00e+00]],
      dtype=float32)

In [24]:
preco_real

array([[  480.],
       [18300.],
       [ 9800.],
       ...,
       [ 9200.],
       [ 3400.],
       [28990.]], dtype=float32)

## Etapa 3: Construção do modelo

In [25]:
len(previsores)

359291

In [27]:
previsores.shape # depois do onehotencoder  => neurônios do input

(359291, 316)

In [31]:
preco_real.shape  #=> neurônios de saída

(359291, 1)

In [29]:
# estimar os números de neurônios das camadas ocultas
(previsores.shape[1] + preco_real.shape[1])/2

158.5

## Etapa 3: Construção do modelo

In [32]:
# Definição da classe do modelo, herdando de nn.Module
#
class regressor_torch(nn.Module):
    def __init__(self):
        super().__init__()                         # inicializa a classe base (nn.Module)
        self.dense0 = nn.Linear(316, 158)          # camada totalmente conectada: 316 entradas ou 316 features → 158 neurônios nas camadas ocultas
        self.dense1 = nn.Linear(158, 158)          # segunda camada: 158 neurônios na primeira camada oculta → 158 neurônios na segunda camada oculta.
        self.dense2 = nn.Linear(158, 1)            # camada final: 158 na segunda camada oculta → 1 saída (previsão)
        self.activation = nn.ReLU()                # função de ativação ReLU (zera valores negativos)

    def forward(self, X):
        X = self.dense0(X)                         # passa os dados pela 1ª camada linear
        X = self.activation(X)                     # aplica ReLU
        X = self.dense1(X)                         # passa pela 2ª camada linear
        X = self.activation(X)                     # aplica ReLU de novo
        X = self.dense2(X)                         # passa pela camada final (gera saída predita)
        return X                                   # retorna a previsão


# Duas camadas escondidas: 158 neurônios cada, com ReLU
# Saída: 1 valor (regressão, ex.: previsão de vendas, preço etc.)


###                                         Camada de ativação

$\begin{array}{|c|c|c|c|}
\hline
\textbf{Função} & \textbf{Fórmula} & \textbf{Intervalo de saída} & \textbf{Uso comum} \\
\hline
\text{Sigmoid} & 
\sigma(x) = \frac{1}{1 + e^{-x}} & (0, 1) & \text{Classificação binária} \\
\hline
\text{Tanh} & 
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} & (-1, 1) & \text{Camadas ocultas (alternativa ao ReLU)} \\
\hline
\text{ReLU} & 
f(x) = \max(0, x) & [0, \infty) & \text{Camadas ocultas (mais usado)} \\
\hline
\text{Leaky ReLU} & 
f(x) = \begin{cases}
x & \text{se } x > 0 \\
\alpha x & \text{se } x \leq 0
\end{cases} & (-\infty, \infty) & \text{Evita neurônios mortos no ReLU} \\
\hline
\text{ELU} & 
f(x) = \begin{cases}
x & \text{se } x > 0 \\
\alpha(e^x - 1) & \text{se } x \leq 0
\end{cases} & (-\alpha, \infty) & \text{Camadas ocultas, alternativa ao ReLU} \\
\hline
\text{Softmax} & 
f(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}} & (0, 1), \ \sum f(x_i) = 1 & \text{Classificação multiclasse} \\
\hline
\end{array}$


🔹 O ReLU (Rectified Linear Unit)

Características:

Simples, rápido de calcular.

Resolve o problema do vanishing gradient (quando os gradientes ficam quase zero em sigmoides/tanh).

Muito usado em camadas ocultas.

⚠️ Problema: pode causar “neurônios mortos” (quando muitos valores ficam ≤ 0 e nunca mais reativam).




In [33]:
regressor_sklearn = NeuralNetRegressor(
    module = regressor_torch,          # minha classe de rede neural (definida antes)
    criterion = torch.nn.L1Loss,       # função de erro → MAE (erro absoluto médio)
    optimizer = torch.optim.Adam,      # otimizador → Adam (ajusta os pesos da rede)
    max_epochs = 100,                  # número máximo de épocas (vezes que o modelo "vê" os dados)
    batch_size = 300,                  # tamanho do lote de dados em cada iteração
    train_split = False                # não divide dados em treino/validação internamente, deixar para o sklearn
)


Resumindo:

NeuralNetRegressor → wrapper que deixa sua rede neural (regressor_torch) com a mesma interface de modelos do scikit-learn (fit, predict, score).

Vai treinar a rede com Adam, otimizando a perda L1Loss (MAE), durante até 100 épocas, em lotes de 300 amostras.

Como train_split=False, ele vai usar todos os dados de treino que você passar (sem separar validação automaticamente).


## Etapa 4: Validação cruzada


1) - Divide os dados em 5 partes (cv=5).

- Treina o modelo em 4 partes.

- Testa na parte restante.

- Repete o processo até todas as partes terem sido usadas como teste.

2) - Calcula a métrica em cada rodada.

- Como você escolheu 'neg_mean_absolute_error', o cross_val_score devolve o MAE negativo (isso é padrão no scikit-learn, porque algumas métricas precisam ser maximizadas).

3) - resultados → será um array com 5 valores, um para cada fold.

In [34]:
resultados = cross_val_score(
    regressor_sklearn,               # meu modelo (NeuralNetRegressor)
    previsores,                      # X → variáveis de entrada (features)
    preco_real,                      # y → variável alvo (valor real a prever) - (target)
    cv = 5,                          # validação cruzada k-fold com 5 divisões
    scoring = 'neg_mean_absolute_error'  # métrica: MAE (erro absoluto médio), mas em valor negativo
)


  epoch    train_loss      dur
-------  ------------  -------
      1     3576.3572  12.3829
      2     3002.0581  15.6682
      3     2856.8516  15.9557
      4     2813.6090  10.6570
      5     2741.3406  14.4750
      6     2750.2524  12.8946
      7     2690.5325  11.0509
      8     2668.2525  15.0420
      9     2683.1201  15.4444
     10     2599.6889  15.0638
     11     2576.4640  8.9591
     12     2557.8768  9.7248
     13     2533.6625  10.0214
     14     2518.3148  16.9227
     15     2508.3104  8.7621
     16     2507.2649  8.5554
     17     2488.9379  16.3574
     18     2479.8670  11.4221
     19     2460.5681  9.5805
     20     2450.9333  9.9913
     21     2476.1944  11.6099
     22     2460.0682  16.4916
     23     2430.4870  15.6230
     24     2424.7273  11.8253
     25     2418.2663  9.8970
     26     2426.1880  8.2794
     27     2419.0641  10.3937
     28     2425.7285  8.9330
     29     2416.7087  8.2381
     30     2405.1933  13.3005
     31     2386.5

In [36]:
media = resultados.mean()
desvio = resultados.std()

print(f'MAE médio {media} e com o desvio {desvio}')

MAE médio -2358.42626953125 e com o desvio 146.52189116046165
